# Liquidity & ALM Risk Engine — full run

LCR · NSFR · liquidity gap & survival horizon · IRRBB (dEVE / dNII).

Runs end to end with no token and no download: the balance sheet is built
deterministically. **Runtime → Run all.**

In [ ]:
import os, sys
if not os.path.exists('src/lcr.py'):
    !git clone https://github.com/ArjiNasim/liquidity-alm-engine.git
    %cd liquidity-alm-engine
!pip -q install scipy matplotlib pyarrow
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import balance_sheet, lcr, nsfr, liquidity_gap, irrbb
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

## 1. The balance sheet

In [ ]:
a, l = balance_sheet.build_balance_sheet()
ta, tl = balance_sheet.balance_check()
print(f'assets {ta:,.0f}M = liabilities+equity {tl:,.0f}M')
a

In [ ]:
l

## 2. Liquidity Coverage Ratio (LCR)

In [ ]:
lr = lcr.lcr_report(a, l)
print(f"Total HQLA      : {lr['total_hqla']:,.0f}M")
print(f"Net 30d outflows: {lr['net_outflows']:,.0f}M")
print(f"LCR             : {lr['lcr_pct']:.0f}%  ({'PASS' if lr['pass'] else 'FAIL'}, min 100%)")
pd.Series(lr['hqla_breakdown'])

## 3. Net Stable Funding Ratio (NSFR)

In [ ]:
nr = nsfr.nsfr_report(a, l)
print(f"ASF  : {nr['asf']:,.0f}M")
print(f"RSF  : {nr['rsf']:,.0f}M")
print(f"NSFR : {nr['nsfr_pct']:.0f}%  ({'PASS' if nr['pass'] else 'FAIL'}, min 100%)")

## 4. Liquidity gap & survival horizon

In [ ]:
sh = liquidity_gap.survival_horizon(a, l, lr['total_hqla'])
print('survival horizon:', sh['survival_horizon'])
ladder = sh['ladder']
fig, ax = plt.subplots(figsize=(9,4))
ax.bar(ladder['bucket'], ladder['marginal_gap']/1000,
       color=['#2e7d32' if v>=0 else '#c0392b' for v in ladder['marginal_gap']], label='marginal gap')
ax.plot(ladder['bucket'], ladder['cumulative_gap']/1000, 'o-', color='#0d2538', label='cumulative gap')
ax.plot(ladder['bucket'], (lr['total_hqla']+ladder['cumulative_gap'])/1000, 's-', color='#b8860b', label='HQLA + cumulative')
ax.axhline(0, color='grey', lw=0.8); ax.legend(); ax.set_ylabel('bn')
ax.set_title('Liquidity gap ladder'); plt.tight_layout(); plt.show()
ladder

## 5. IRRBB — dEVE / dNII under Basel shocks

In [ ]:
ir = irrbb.irrbb_report(a, l, tier1_capital=17_000)
print(f"dNII +100bp : {ir['delta_nii_up_100bp']:,.0f}M")
print(f"dNII -100bp : {ir['delta_nii_down_100bp']:,.0f}M")
print(f"worst dEVE  : {ir['worst_eve']:,.0f}M ({ir['worst_eve_scenario']}, {ir['worst_eve_pct_tier1']:.1f}% of Tier 1)")
print(f"SOT breach  : {ir['sot_breach']}")
ir['delta_eve']

In [ ]:
ir['delta_eve'].set_index('scenario')['delta_eve'].plot(
    kind='bar', figsize=(8,3.5), color='#164e63', title='dEVE by Basel shock (M)')
plt.axhline(0, color='grey', lw=0.8); plt.tight_layout(); plt.show()

## 6. Save artefact

In [ ]:
import os; os.makedirs('data', exist_ok=True)
a.to_parquet('data/balance_sheet.parquet')
print('saved data/balance_sheet.parquet')
print('\nLiquidity & ALM engine run complete.')